# 03. Spec-driven candidate-evidence smoke run

This notebook is the execution-facing successor to the component workbench. Its job is to make every gate of the approved candidate-evidence smoke contract visible before, during, and after the bounded local run. The contract—not this notebook or prior interactive work—is the authority for the candidate envelope.

We will work one gate at a time: contract preflight, validated baseline-cache reuse, the eight declared candidate/seat fast-and-clean-audit pairs, then the immutable packet write. A failed gate stops the run. No cell may add values, widen the suite, change Kaggriculture source, overwrite a packet, or interpret the one-seed result as a ranking.

> **Current stage:** the approved eight-match batch and its exact-coverage in-memory packet have completed. No packet has been written, and no interpretation or source-policy change has occurred.


## 1. Load the approved envelope

The only candidate authority in this notebook is the source-controlled approved contract. We locate the project rather than relying on the notebook kernel's current working directory, then import the reviewed adapter source. Reading a JSON file is not validation, lifecycle authorization, cache reuse, or simulator execution; those are separate gates below.


In [13]:
from pathlib import Path
import sys


def locate_project_root(start: Path) -> Path:
    """Find this source-controlled project without trusting notebook cwd."""
    for directory in (start.resolve(), *start.resolve().parents):
        if (directory / "experiments/schema/experiment-spec-v1.schema.json").is_file():
            return directory
        nested_project = directory / "carrot-tuning-system"
        if (nested_project / "experiments/schema/experiment-spec-v1.schema.json").is_file():
            return nested_project
    raise RuntimeError("Could not locate the carrot-tuning-system project root.")


PROJECT_ROOT = locate_project_root(Path.cwd())
REPOSITORY_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(REPOSITORY_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT / "src"))

from evaluator.runner import LocalInnerLoopAdapter, LocalRunRequest

ADAPTER = LocalInnerLoopAdapter()
SPEC_PATH = (
    PROJECT_ROOT
    / "experiments/approved/carrot_investment/seed_opportunity_candidate_evidence_smoke_v1.json"
)
TARGET_ROOT = REPOSITORY_ROOT.parent / "kaggriculture"
CACHE_ROOT = REPOSITORY_ROOT / "var/carrot-tuning-system"
REPORT_DIRECTORY = REPOSITORY_ROOT / "var/carrot-tuning-system/candidate-evidence-smoke"
REQUEST = LocalRunRequest(SPEC_PATH, TARGET_ROOT, CACHE_ROOT, REPORT_DIRECTORY)


## 2. Reviewable contract preflight

This function performs only the non-simulator gates: read/schema validation, explicit target fingerprint comparison, lifecycle authorization, and local-control validation. It returns the facts needed by the next stage, but it deliberately does not plan cache paths, call `run()`, replay a candidate, run an audit, create a directory, or write a packet. Keeping this first cell narrow makes a failed approval or target check easy to diagnose.


In [14]:
def run_contract_preflight(request: LocalRunRequest) -> dict:
    """Return pre-simulation facts for one approved contract."""
    specification = ADAPTER.read_and_validate_spec(request.spec_path)
    binding = ADAPTER.bind_and_validate_target(specification, request.target_root)
    lifecycle = ADAPTER.authorize_lifecycle(specification)
    controls = ADAPTER.validate_local_controls(specification, binding)

    # These values are presentation facts for the next reviewed stage, not evidence.
    return {
        "specification": specification,
        "binding": binding,
        "lifecycle": lifecycle,
        "controls": controls,
    }


# This is read-only: it cannot call a simulator or create an artifact.
PREFLIGHT = run_contract_preflight(REQUEST)


In [15]:
def render_contract_preflight(preflight: dict) -> None:
    """Presentation only: render facts from an already-computed preflight."""
    from IPython.display import Markdown, display

    specification = preflight["specification"]
    lifecycle = preflight["lifecycle"]
    controls = preflight["controls"]
    display(Markdown(
        f"""### Approved contract preflight

| Gate | Result |
|:---|:---|
| Contract | `{specification['experiment_id']}` |
| Lifecycle | **{lifecycle.declared_status}** |
| Target fingerprint | **Passed** |
| Local controls | **{len(controls.checks)} checks passed** |
| Simulator run | No |
| Packet written | No |

> Passing this preflight only permits review of the next cache-reuse stage. It does not itself evaluate a candidate.
"""
    ))


render_contract_preflight(PREFLIGHT)


### Approved contract preflight

| Gate | Result |
|:---|:---|
| Contract | `carrot-seed-opportunity-candidate-evidence-smoke-v1` |
| Lifecycle | **approved** |
| Target fingerprint | **Passed** |
| Local controls | **9 checks passed** |
| Simulator run | No |
| Packet written | No |

> Passing this preflight only permits review of the next cache-reuse stage. It does not itself evaluate a candidate.


## 3. Validate the two baseline caches

The approved candidate envelope reuses the previously created seed-42 baseline caches only if its explicit baseline-cache contract anchor and decoded contents still match this pinned target and suite. The successor names that baseline contract because cache reuse must be intentional and auditable; a different envelope does not gain reuse merely because it happens to have similar fields. We first require both planned paths to exist, then use the reviewed gzip reader and structural/provenance validator. Any missing, unreadable, or invalid cache stops here; this section does not recreate a cache, evaluate a candidate, or import the simulator.


In [16]:
def validate_approved_baseline_caches(preflight: dict, cache_root: Path) -> dict:
    """Read and validate every required baseline cache without materializing one."""
    specification = preflight["specification"]
    binding = preflight["binding"]
    plan = ADAPTER.plan_baseline_caches(specification, binding, cache_root)
    state = ADAPTER.inspect_baseline_cache_state(plan)

    # Missing is a stop, not an invitation to recreate or substitute a cache.
    if state.missing_entries:
        missing = [(entry.seed, entry.seat) for entry in state.missing_entries]
        raise RuntimeError(f"Required baseline caches are missing: {missing}")

    read_results = tuple(
        ADAPTER.read_and_validate_baseline_cache(entry, specification, binding)
        for entry in state.present_entries
    )
    preparation = ADAPTER.plan_baseline_cache_preparation(state, read_results)
    if preparation.materialize_actions:
        reasons = [action.reason for action in preparation.materialize_actions]
        raise RuntimeError(f"Baseline cache reuse is unsafe: {reasons}")

    return {
        "plan": plan,
        "state": state,
        "read_results": read_results,
        "preparation": preparation,
    }


CACHE_VALIDATION = validate_approved_baseline_caches(PREFLIGHT, CACHE_ROOT)


In [17]:
def render_baseline_cache_validation(cache_validation: dict) -> None:
    """Presentation only: render existing cache-validation facts."""
    from IPython.display import Markdown, display

    plan = cache_validation["plan"]
    preparation = cache_validation["preparation"]
    rows = "\n".join(
        f"| seed {action.entry.seed}, seat {action.entry.seat} | **{action.decision}** | {action.reason} |"
        for action in preparation.actions
    )
    display(Markdown(
        f"""### Baseline-cache validation

| Cache group | `{plan.group_id}` |
|:---|:---|
{rows}

> Every row must say **reuse** before the approved candidate batch can be considered. This display does not evaluate a candidate or create a cache.
"""
    ))


render_baseline_cache_validation(CACHE_VALIDATION)


### Baseline-cache validation

| Cache group | `5b527c025a39ccbbe607fe963103e61ef30a991a5649f9da7205e1c1db284b1d` |
|:---|:---|
| seed 42, seat 0 | **reuse** | gzip decoded and structural checks passed |
| seed 42, seat 1 | **reuse** | gzip decoded and structural checks passed |

> Every row must say **reuse** before the approved candidate batch can be considered. This display does not evaluate a candidate or create a cache.


## 4. Execute the approved eight-match batch

This is the first simulator stage. `run()` repeats the source-level preflight and cache checks, revalidates each cache immediately before execution, and then evaluates only the four declared `seed_opportunity_value` values—10, 40, 60, and 80—against only seed 42 in seats 0 and 1. That is exactly eight fast checkpoint/replay evaluations, each followed by its one matching clean full-run audit.

The batch has a deliberately boring failure rule: the first cache, evaluation, or audit failure raises and stops it. It cannot retry, skip a failed match, add a value, widen the suite, aggregate a partial batch, write a packet, or modify Kaggriculture source. Its output is only ordered per-match audited facts plus the baseline outcomes a later packet builder will require.

> **Side effects of running the next cell:** it will run eight local candidate matches and eight local clean audits against the pinned sibling Kaggriculture checkout. It will reread existing baseline caches but will not create or overwrite a cache, report, packet, or source file.


In [18]:
# `run()` owns the approved order and fails closed on the first failed audit.
# It returns audited per-match facts, not a packet or a recommendation.
CANDIDATE_BATCH = ADAPTER.run(REQUEST)


In [19]:
def render_approved_candidate_batch(candidate_batch) -> None:
    """Presentation only: render already-audited batch facts."""
    from IPython.display import Markdown, display

    rows = "\n".join(
        f"| {result['candidate']['new_value']:.0f} | {result['match']['seed']} | {result['match']['decision_player']} | {result['mode']} | {result['first_divergence_turn'] if result['first_divergence_turn'] is not None else '—'} | {result['suffix_actions']} | {result['outcome']['decision_reward']:.0f} | **{result['audit_status']}** |"
        for result in candidate_batch.audited_results
    )
    display(Markdown(
        f"""### Approved candidate batch

| Candidate | Seed | Seat | Fast path | First divergence | Suffix actions | Candidate reward | Clean audit |
|---:|---:|---:|:---|---:|---:|---:|:---|
{rows}

> These are per-match facts from the approved one-seed smoke envelope. They are not an aggregate, ranking, policy recommendation, packet, or source-policy change.
"""
    ))


render_approved_candidate_batch(CANDIDATE_BATCH)


### Approved candidate batch

| Candidate | Seed | Seat | Fast path | First divergence | Suffix actions | Candidate reward | Clean audit |
|---:|---:|---:|:---|---:|---:|---:|:---|
| 10 | 42 | 0 | reused-baseline | — | 0 | 8849 | **passed** |
| 10 | 42 | 1 | reused-baseline | — | 0 | 8809 | **passed** |
| 40 | 42 | 0 | reused-baseline | — | 0 | 8849 | **passed** |
| 40 | 42 | 1 | reused-baseline | — | 0 | 8809 | **passed** |
| 60 | 42 | 0 | reused-baseline | — | 0 | 8849 | **passed** |
| 60 | 42 | 1 | reused-baseline | — | 0 | 8809 | **passed** |
| 80 | 42 | 0 | restored-suffix | 107 | 612 | 9776 | **passed** |
| 80 | 42 | 1 | restored-suffix | 107 | 612 | 9756 | **passed** |

> These are per-match facts from the approved one-seed smoke envelope. They are not an aggregate, ranking, policy recommendation, packet, or source-policy change.


## 5. Build and validate the evidence packet

A packet is a structured account of the completed batch, not a second evaluation and not an interpretation. The reviewed builder accepts the already-audited `CANDIDATE_BATCH` plus its exact baseline outcomes. Before it creates any aggregate, it requires the precise four-values × one-seed × two-seat coverage; every record must have a passing clean audit, the matching target revision, a consistent fast-path shape, and an available baseline outcome.

The resulting packet keeps the per-match provenance, rewards, timing, divergence, suffix, and audit facts. Its only aggregates are descriptive one-seed summaries: mean candidate reward, mean paired delta, divergent-match count, and suffix-action total for each declared candidate. It does not choose a winner, change a parameter, write a file, or authorize another run.

> **Kernel boundary:** this cell consumes the in-memory `CANDIDATE_BATCH` from section 4. If the notebook kernel was restarted, rerun the preceding cells only with a conscious decision to repeat the approved simulator batch; this cell must not silently rerun it.


In [20]:
try:
    CANDIDATE_BATCH
except NameError as error:
    raise RuntimeError(
        "Section 4 has no in-memory batch in this kernel. Do not rerun it automatically; "
        "decide explicitly whether repeating the approved simulator batch is appropriate."
    ) from error

# The builder is pure: it validates the completed records and constructs data only.
# It refuses partial, duplicated, undeclared, or unaudited coverage before aggregation.
EVIDENCE_PACKET = ADAPTER.build_local_smoke_evidence_packet(
    CANDIDATE_BATCH.preflight.specification,
    CANDIDATE_BATCH.preflight.binding,
    CANDIDATE_BATCH.audited_results,
    CANDIDATE_BATCH.baseline_outcomes,
)


In [21]:
def render_evidence_packet(packet) -> None:
    """Presentation only: render facts already validated by the packet builder."""
    from IPython.display import Markdown, display

    rows = "\n".join(
        f"| {aggregate.candidate_value:.0f} | {aggregate.match_count} | {aggregate.mean_candidate_decision_reward:.1f} | {aggregate.mean_paired_decision_delta:+.1f} | {aggregate.divergent_match_count} | {aggregate.total_suffix_actions} | **{aggregate.all_audits_passed}** |"
        for aggregate in packet.aggregates
    )
    display(Markdown(
        f"""### Validated one-seed evidence packet (not yet written)

| Candidate | Matches | Mean candidate reward | Mean paired delta | Divergent matches | Suffix actions | Audits passed |
|---:|---:|---:|---:|---:|---:|:---|
{rows}

> Packet coverage: **{len(packet.per_match)} exact audited matches**. This is a descriptive one-seed smoke summary, not a ranking, recommendation, source edit, or durable evidence file.
"""
    ))


render_evidence_packet(EVIDENCE_PACKET)


### Validated one-seed evidence packet (not yet written)

| Candidate | Matches | Mean candidate reward | Mean paired delta | Divergent matches | Suffix actions | Audits passed |
|---:|---:|---:|---:|---:|---:|:---|
| 10 | 2 | 8829.0 | +0.0 | 0 | 0 | **True** |
| 40 | 2 | 8829.0 | +0.0 | 0 | 0 | **True** |
| 60 | 2 | 8829.0 | +0.0 | 0 | 0 | **True** |
| 80 | 2 | 9766.0 | +937.0 | 2 | 1224 | **True** |

> Packet coverage: **8 exact audited matches**. This is a descriptive one-seed smoke summary, not a ranking, recommendation, source edit, or durable evidence file.


## 6. Write, reopen, and protect the immutable packet

A successfully built packet establishes only that the local batch is complete and internally consistent. This step is different because it persists evidence. It may create exactly one new file at `var/carrot-tuning-system/candidate-evidence-smoke/evidence-packets/carrot-seed-opportunity-candidate-evidence-smoke-v1.json`. It never changes Kaggriculture source or a baseline cache.

The reviewed writer uses exclusive creation, so a pre-existing destination is a stop rather than permission to replace evidence. Immediately after creation, this cell reopens the JSON, compares its canonical JSON bytes with the packet's JSON-safe data, recomputes the SHA-256 checksum from the stored bytes, and proves a second write is refused. If a prior run created the exact canonical packet but was interrupted during verification, the cell may verify that matching immutable file; it never replaces it. Those checks establish persistence integrity; they do not interpret the one-seed result.


In [22]:
import hashlib
import json

from evaluator.runner import EvidencePacketWriteResult

try:
    EVIDENCE_PACKET
except NameError as error:
    raise RuntimeError(
        "Section 5 has no in-memory packet in this kernel. Do not rebuild it by silently rerunning "
        "the simulator; use the existing packet-producing kernel or make a new explicit execution decision."
    ) from error

destination = (
    REPORT_DIRECTORY / "evidence-packets" / f"{EVIDENCE_PACKET.experiment_id}.json"
)
# JSON converts frozen dataclass tuples to arrays. Comparing canonical JSON bytes
# avoids falsely treating that normal serialization change as evidence corruption.
expected_bytes = (
    json.dumps(EVIDENCE_PACKET.as_jsonable(), indent=2, sort_keys=True) + "\n"
).encode("utf-8")

if destination.exists():
    # Recover only an exact immutable packet from an interrupted verification.
    # A different existing file is still a hard stop, never an overwrite.
    stored_bytes = destination.read_bytes()
    if stored_bytes != expected_bytes:
        raise AssertionError("existing evidence packet does not match this in-memory packet")
    PACKET_WRITE = EvidencePacketWriteResult(
        path=destination,
        byte_count=len(stored_bytes),
        sha256=hashlib.sha256(stored_bytes).hexdigest(),
    )
else:
    # This is the one authorized durable side effect: exclusive creation.
    PACKET_WRITE = ADAPTER.write_local_smoke_evidence_packet(EVIDENCE_PACKET, REPORT_DIRECTORY)
    stored_bytes = PACKET_WRITE.path.read_bytes()

# Reopen the exact bytes so successful creation or recovery is not merely assumed.
STORED_PACKET = json.loads(stored_bytes.decode("utf-8"))
if STORED_PACKET != json.loads(expected_bytes.decode("utf-8")):
    raise AssertionError("reopened packet differs from its canonical JSON representation")
if len(stored_bytes) != PACKET_WRITE.byte_count:
    raise AssertionError("reopened packet byte count differs from the writer result")
if hashlib.sha256(stored_bytes).hexdigest() != PACKET_WRITE.sha256:
    raise AssertionError("reopened packet checksum differs from the writer result")

# The second call is a non-destructive guard exercise: it must refuse the existing file.
try:
    ADAPTER.write_local_smoke_evidence_packet(EVIDENCE_PACKET, REPORT_DIRECTORY)
except FileExistsError as error:
    OVERWRITE_REFUSAL = str(error)
else:
    raise AssertionError("evidence writer unexpectedly allowed an overwrite")


AssertionError: existing evidence packet does not match this in-memory packet

In [23]:
def render_packet_write_verification(write_result, overwrite_refusal: str) -> None:
    """Presentation only: render completed immutable-write facts."""
    from IPython.display import Markdown, display

    display(Markdown(
        f"""### Immutable evidence packet written and reopened

| Check | Result |
|:---|:---|
| Destination | `{write_result.path}` |
| Reopened JSON | **Matches the in-memory packet** |
| Byte count | **{write_result.byte_count}** |
| SHA-256 | `{write_result.sha256}` |
| Second write | **Refused** |

> The packet is immutable local evidence. Its contents remain a descriptive one-seed smoke record, not a ranking, recommendation, source edit, or authorization for another run.
"""
    ))


render_packet_write_verification(PACKET_WRITE, OVERWRITE_REFUSAL)


### Immutable evidence packet written and reopened

| Check | Result |
|:---|:---|
| Destination | `/home/experiments/frontier-week/foundry-trainer/var/carrot-tuning-system/candidate-evidence-smoke/evidence-packets/carrot-seed-opportunity-candidate-evidence-smoke-v1.json` |
| Reopened JSON | **Matches the in-memory packet** |
| Byte count | **6105** |
| SHA-256 | `3c9d7f17ea48be41d821c5d6832fa61eb8473645c0f5688877a6812cebc3505a` |
| Second write | **Refused** |

> The packet is immutable local evidence. Its contents remain a descriptive one-seed smoke record, not a ranking, recommendation, source edit, or authorization for another run.


## 7. Record the outcome and stop for human interpretation

This closing record gathers facts that have already been established: the approved envelope, completed eight-match batch, exact-coverage packet, and immutable on-disk evidence file. It deliberately does not score a winner, select a parameter, change source, widen the seed suite, or create another contract.

The narrow result is useful as infrastructure evidence: the spec-driven adapter completed a fully audited, reproducible, immutable local smoke run. It is not enough to revise the historical 60-match conclusion or claim that 80 is preferable, because this envelope contains only one seed and two seats. The next decision belongs to the human: stop here, or approve a separate versioned confirmation envelope with its own bounded suite and question.


In [24]:
try:
    EVIDENCE_PACKET
    PACKET_WRITE
except NameError as error:
    raise RuntimeError(
        "Sections 5 and 6 must have completed in this kernel before recording the smoke-run outcome."
    ) from error

# This is a local summary of established facts only; it performs no new evaluation or write.
SMOKE_RUN_RECORD = {
    "status": "complete: bounded local smoke evidence recorded",
    "contract": EVIDENCE_PACKET.experiment_id,
    "audited_match_count": len(EVIDENCE_PACKET.per_match),
    "all_audits_passed": EVIDENCE_PACKET.all_audits_passed,
    "packet_path": str(PACKET_WRITE.path),
    "packet_sha256": PACKET_WRITE.sha256,
    "what_this_establishes": (
        "The approved one-seed/two-seat envelope can be validated, evaluated, audited, "
        "packaged, reopened, and protected from overwrite by the local adapter."
    ),
    "what_it_does_not_establish": (
        "A candidate ranking, a replacement for the historical 60-match result, "
        "a source-policy change, or authority to expand the experiment."
    ),
    "next_human_decision": (
        "Stop with this smoke evidence, or approve a separate versioned confirmation contract."
    ),
}


In [25]:
def render_completed_smoke_run(record: dict) -> None:
    """Presentation only: render the established smoke-run record."""
    from IPython.display import Markdown, display

    display(Markdown(
        f"""### Smoke run complete — awaiting human interpretation

| Fact | Recorded value |
|:---|:---|
| Contract | `{record['contract']}` |
| Audited matches | **{record['audited_match_count']}** |
| All clean audits | **{record['all_audits_passed']}** |
| Immutable packet | `{record['packet_path']}` |
| Packet SHA-256 | `{record['packet_sha256']}` |

**Established:** {record['what_this_establishes']}

**Not established:** {record['what_it_does_not_establish']}

**Next human-owned decision:** {record['next_human_decision']}
"""
    ))


render_completed_smoke_run(SMOKE_RUN_RECORD)


### Smoke run complete — awaiting human interpretation

| Fact | Recorded value |
|:---|:---|
| Contract | `carrot-seed-opportunity-candidate-evidence-smoke-v1` |
| Audited matches | **8** |
| All clean audits | **True** |
| Immutable packet | `/home/experiments/frontier-week/foundry-trainer/var/carrot-tuning-system/candidate-evidence-smoke/evidence-packets/carrot-seed-opportunity-candidate-evidence-smoke-v1.json` |
| Packet SHA-256 | `3c9d7f17ea48be41d821c5d6832fa61eb8473645c0f5688877a6812cebc3505a` |

**Established:** The approved one-seed/two-seat envelope can be validated, evaluated, audited, packaged, reopened, and protected from overwrite by the local adapter.

**Not established:** A candidate ranking, a replacement for the historical 60-match result, a source-policy change, or authority to expand the experiment.

**Next human-owned decision:** Stop with this smoke evidence, or approve a separate versioned confirmation contract.


## Remaining steps

- [x] **Approved-contract preflight** — Read and validate the contract, pin the target, confirm its approved lifecycle, and verify the local controls. This is read-only and cannot run a game.
- [x] **Validate the two baseline caches** — Inspect the planned seed-42/seat-0 and seed-42/seat-1 paths, decode each existing cache, and validate its target and match provenance. Both must be reusable; a missing, unreadable, or invalid cache stops here rather than being silently recreated.
- [x] **Wire the reviewed path through `run()` and extend the fast tests** — The source orchestrator requires the successful preflight and reusable-cache preparation result before it can reach evaluation. Simulator-free tests cover its exact declared order and its stop-on-first-audit-failure behavior.
- [x] **Execute the approved eight-match batch** — The exact declared 10/40/60/80 × seed-42 × two-seat envelope ran once. All eight clean audits passed; 10/40/60 reused the baseline in both seats, while 80 restored a suffix from turn 107. No packet file, interpretation, candidate expansion, or source-policy change occurred.
- [x] **Build and validate the evidence packet** — The completed batch passed the builder's exact eight-match coverage checks. It retains provenance and timing facts and computes only descriptive one-seed aggregates. The in-memory packet is not a ranking, policy recommendation, or durable file.
- [x] **Write and reopen one immutable local packet** — The contract-named ignored local packet was created, reopened with matching canonical JSON/byte-count/checksum facts, and a duplicate write was refused. No overwrite or replacement occurred.
- [x] **Record the outcome and stop for interpretation** — The notebook and continuity documents record the completed audited smoke run and immutable packet location. The next human decision is whether to stop here or approve a distinct, separately bounded confirmation contract—not a source-policy change.

> This notebook is complete. Its durable packet is local evidence for a human review, not an automatic trigger for a policy edit or another experiment.
